# 花卉图片分类器：Keras 训练并导出 TFLite

> 实验5-1：TensorFlow 模型生成
> 基于教程：https://blog.csdn.net/llfjfz/article/details/161630612

## 流程

```
下载数据集 → 构建 MobileNetV2 模型 → 训练 5 epoch → 评估 → 导出 .tflite + labels.txt → 冒烟测试
```

## 环境

- Python 3.13
- TensorFlow >= 2.15
- 推荐在 venv 中运行

In [1]:
# Cell 1: 安装依赖（仅在首次运行时取消注释）
# %pip install tensorflow>=2.15 matplotlib>=3.7 numpy>=1.23

In [2]:
# Cell 2: 导入库并设置参数
import tarfile
from pathlib import Path

import numpy as np
import tensorflow as tf

# TensorFlow 官方花卉数据集
FLOWER_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

print("TensorFlow 版本:", tf.__version__)

# ── 参数配置 ──
# DATA_DIR = None：自动下载并使用 TensorFlow 官方 flowers 数据集
# DATA_DIR = r"D:\path\to\my_images"：使用你自己的图片分类目录（每类一个子文件夹）
DATA_DIR = None

# 导出目录
EXPORT_DIR = "exported_flower_model"

# 训练参数
EPOCHS = 5
BATCH_SIZE = 32
IMAGE_SIZE = 224
LEARNING_RATE = 1e-3

# TFLite 量化方式: dynamic / float16 / int8 / none
QUANTIZATION = "dynamic"

# 固定随机种子
SEED = 123

print(f"EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, IMAGE_SIZE={IMAGE_SIZE}")
print(f"LEARNING_RATE={LEARNING_RATE}, QUANTIZATION={QUANTIZATION}")

C:\Users\33525\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


TensorFlow 版本: 2.21.0
EPOCHS=5, BATCH_SIZE=32, IMAGE_SIZE=224
LEARNING_RATE=0.001, QUANTIZATION=dynamic


In [3]:
# Cell 3: 加载并划分数据集

def load_flower_datasets(data_dir, image_size, batch_size, seed):
    """加载花卉数据集，返回 train/val/test 三个 tf.data.Dataset 和类别名称列表。"""
    
    if data_dir is None:
        # 下载 TensorFlow 官方 flower_photos 数据集
        archive_path = tf.keras.utils.get_file(
            "flower_photos.tgz",
            FLOWER_URL,
            extract=False,
        )
        archive_path = Path(archive_path)

        # 检查是否已解压
        candidates = [
            archive_path.parent / "flower_photos",
            archive_path.parent / "flower_photos_extracted" / "flower_photos",
        ]
        data_dir = next((path for path in candidates if path.exists()), None)
        if data_dir is None:
            print("正在解压数据集...")
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(archive_path.parent / "flower_photos_extracted")
            data_dir = archive_path.parent / "flower_photos_extracted" / "flower_photos"
    else:
        data_dir = Path(data_dir)

    # 从目录读取图片，按子文件夹名生成标签
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=(image_size, image_size),
        batch_size=batch_size,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=(image_size, image_size),
        batch_size=batch_size,
    )
    class_names = train_ds.class_names

    # 从验证集中分出一半作为测试集
    val_batches = int(tf.data.experimental.cardinality(val_ds).numpy())
    test_ds = val_ds.take(val_batches // 2)
    val_ds = val_ds.skip(val_batches // 2)

    # 缓存、打乱（仅训练集）、预取加速
    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000, seed=seed).prefetch(autotune)
    val_ds = val_ds.cache().prefetch(autotune)
    test_ds = test_ds.cache().prefetch(autotune)

    return train_ds, val_ds, test_ds, class_names


# 执行加载
train_ds, val_ds, test_ds, class_names = load_flower_datasets(
    DATA_DIR, IMAGE_SIZE, BATCH_SIZE, SEED
)

print(f"类别数量: {len(class_names)}")
print(f"类别名称: {class_names}")

Found 3670 files belonging to 5 classes.


Using 2936 files for training.


Found 3670 files belonging to 5 classes.


Using 734 files for validation.


类别数量: 5
类别名称: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [4]:
# Cell 4: 构建 MobileNetV2 迁移学习模型

def build_model(num_classes, image_size, learning_rate):
    """构建基于 MobileNetV2 的图像分类模型（迁移学习）。"""
    
    inputs = tf.keras.Input(shape=(image_size, image_size, 3), name="image")

    # MobileNetV2 预处理：像素值转换到模型期望的范围
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

    # 加载 ImageNet 预训练的 MobileNetV2（去掉原有的 1000 类分类头）
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(image_size, image_size, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )

    # 冻结预训练参数，只训练新增的分类层
    base_model.trainable = False
    x = base_model(x, training=False)
    x = tf.keras.layers.Dropout(0.2)(x)

    # 新分类头：Dense(num_classes, softmax)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


# 创建模型（首次运行会下载 MobileNetV2 的 ImageNet 权重，约 14MB）
model = build_model(len(class_names), IMAGE_SIZE, LEARNING_RATE)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
# Cell 5: 训练模型

print(f"开始训练，共 {EPOCHS} 个 epoch...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

开始训练，共 5 个 epoch...
Epoch 1/5


 1/92 ━━━━━━━━━━━━━━━━━━━━ 5:06 3s/step - accuracy: 0.1875 - loss: 1.9981

 2/92 ━━━━━━━━━━━━━━━━━━━━ 22s 246ms/step - accuracy: 0.2031 - loss: 2.0089

 3/92 ━━━━━━━━━━━━━━━━━━━━ 22s 253ms/step - accuracy: 0.2118 - loss: 1.9750

 4/92 ━━━━━━━━━━━━━━━━━━━━ 21s 249ms/step - accuracy: 0.2194 - loss: 1.9356

...

89/92 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.9221 - loss: 0.2333

90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.9220 - loss: 0.2334

91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.9220 - loss: 0.2336

92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.9220 - loss: 0.2337

92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 269ms/step - accuracy: 0.9186 - loss: 0.2471 - val_accuracy: 0.9136 - val_loss: 0.2762


In [6]:
# Cell 6: 评估模型

loss, accuracy = model.evaluate(test_ds)
print(f"\n===== 测试集评估结果 =====")
print(f"test_loss   = {loss:.4f}")
print(f"test_accuracy = {accuracy:.4f} ({accuracy*100:.2f}%)")

 1/11 ━━━━━━━━━━━━━━━━━━━━ 3s 302ms/step - accuracy: 0.8750 - loss: 0.2512

 2/11 ━━━━━━━━━━━━━━━━━━━━ 2s 247ms/step - accuracy: 0.8516 - loss: 0.3216

 3/11 ━━━━━━━━━━━━━━━━━━━━ 1s 247ms/step - accuracy: 0.8559 - loss: 0.3276

 4/11 ━━━━━━━━━━━━━━━━━━━━ 1s 251ms/step - accuracy: 0.8626 - loss: 0.3269

 5/11 ━━━━━━━━━━━━━━━━━━━━ 1s 252ms/step - accuracy: 0.8689 - loss: 0.3252

...

 9/11 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.8771 - loss: 0.3316

10/11 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - accuracy: 0.8794 - loss: 0.3292

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step - accuracy: 0.8811 - loss: 0.3280

11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 251ms/step - accuracy: 0.8977 - loss: 0.3163



===== 测试集评估结果 =====
test_loss   = 0.3163
test_accuracy = 0.8977 (89.77%)


In [7]:
# Cell 7: TFLite 模型转换函数

def convert_to_tflite(model, quantization, representative_ds=None):
    """将 Keras 模型转换为 TFLite 格式，支持多种量化方式。"""
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantization == "dynamic":
        # 动态范围量化：最常用、最容易成功的压缩方式
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif quantization == "float16":
        # float16 量化
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif quantization == "int8":
        # int8 全整数量化：需要代表性数据集校准
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        def representative_data_gen():
            for images, _ in representative_ds.take(100):
                for image in images:
                    yield [tf.expand_dims(tf.cast(image, tf.float32), 0)]

        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
    elif quantization != "none":
        raise ValueError(f"Unsupported quantization mode: {quantization}")

    return converter.convert()


print("TFLite 转换函数已定义")

TFLite 转换函数已定义


In [8]:
# Cell 8: 导出模型文件

export_dir = Path(EXPORT_DIR)
export_dir.mkdir(parents=True, exist_ok=True)

# 8.1 保存标签文件
labels_path = export_dir / "labels.txt"
labels_path.write_text("\n".join(class_names) + "\n", encoding="utf-8")
print(f"已保存标签文件: {labels_path}")
print(f"  内容: {class_names}")

# 8.2 保存 Keras 原始模型
keras_path = export_dir / "flower_classifier.keras"
model.save(keras_path)
print(f"已保存 Keras 模型: {keras_path}")

# 8.3 转换并保存 TFLite 模型
print(f"正在转换 TFLite 模型 (量化方式: {QUANTIZATION})...")
tflite_model = convert_to_tflite(model, QUANTIZATION, train_ds)
tflite_path = export_dir / "model.tflite"
tflite_path.write_bytes(tflite_model)

# 打印模型大小
keras_size_mb = keras_path.stat().st_size / (1024 * 1024)
tflite_size_mb = tflite_path.stat().st_size / (1024 * 1024)
print(f"已保存 TFLite 模型: {tflite_path}")
print(f"\n===== 模型大小对比 =====")
print(f"Keras 模型:   {keras_size_mb:.1f} MB")
print(f"TFLite 模型:  {tflite_size_mb:.1f} MB")
print(f"压缩比:       {keras_size_mb / tflite_size_mb:.1f}x")

已保存标签文件: exported_flower_model\labels.txt
  内容: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


已保存 Keras 模型: exported_flower_model\flower_classifier.keras
正在转换 TFLite 模型 (量化方式: dynamic)...


INFO:tensorflow:Assets written to: C:\Users\33525\AppData\Local\Temp\tmpu_sytiaf\assets


INFO:tensorflow:Assets written to: C:\Users\33525\AppData\Local\Temp\tmpu_sytiaf\assets


Saved artifact at 'C:\Users\33525\AppData\Local\Temp\tmpu_sytiaf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2357964471952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357964471760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971059152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357964471184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357964472144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971059728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971058768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971060304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971058960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357971059536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  235797105992

已保存 TFLite 模型: exported_flower_model\model.tflite

===== 模型大小对比 =====
Keras 模型:   9.3 MB
TFLite 模型:  2.4 MB
压缩比:       3.8x


In [9]:
# Cell 9: 冒烟测试——验证导出的 TFLite 模型能否正常推理

def smoke_test_tflite(tflite_path, test_ds, class_names):
    """用测试图片快速验证 TFLite 模型。"""
    
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # 取 8 张测试图片
    images, labels = next(iter(test_ds.unbatch().batch(8)))
    input_data = tf.cast(images, input_details["dtype"]).numpy()

    # uint8 量化模型需要特殊处理
    if input_details["dtype"] == np.uint8:
        scale, zero_point = input_details["quantization"]
        if scale:
            input_data = images.numpy() / scale + zero_point
            input_data = np.clip(input_data, 0, 255).astype(np.uint8)

    predictions = []
    for image in input_data:
        interpreter.set_tensor(input_details["index"], np.expand_dims(image, 0))
        interpreter.invoke()
        predictions.append(interpreter.get_tensor(output_details["index"])[0])

    predicted_ids = np.argmax(np.asarray(predictions), axis=1)
    print(f"\n===== TFLite 冒烟测试（前 5 张） =====")
    correct = 0
    for i, (expected, predicted) in enumerate(zip(labels.numpy()[:8], predicted_ids[:8])):
        is_correct = expected == predicted
        if is_correct:
            correct += 1
        mark = "✓" if is_correct else "✗"
        print(f"  [{mark}] 真实={class_names[expected]:12s}  预测={class_names[predicted]}")
    print(f"\n准确率: {correct}/8 = {correct/8*100:.1f}%")
    print("TFLite 模型推理正常！")


# 执行冒烟测试
smoke_test_tflite(tflite_path, test_ds, class_names)

C:\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



===== TFLite 冒烟测试（前 5 张） =====
  [✓] 真实=daisy         预测=daisy
  [✓] 真实=tulips        预测=tulips
  [✓] 真实=sunflowers    预测=sunflowers
  [✓] 真实=daisy         预测=daisy
  [✓] 真实=sunflowers    预测=sunflowers
  [✓] 真实=tulips        预测=tulips
  [✓] 真实=dandelion     预测=dandelion
  [✓] 真实=roses         预测=roses

准确率: 8/8 = 100.0%
TFLite 模型推理正常！


## 完成

导出目录 `exported_flower_model/` 中包含：

| 文件 | 用途 |
|------|------|
| `model.tflite` | 可部署到 Android 的 TFLite 量化模型 |
| `labels.txt` | 5 类花卉标签（每行一个） |
| `flower_classifier.keras` | 原始 Keras 模型（可重新训练/微调） |

### 在实验四的 APP 中验证

1. 将 `model.tflite` 重命名为 `FlowerModel.tflite`
2. 替换 `start/src/main/ml/FlowerModel.tflite`
3. 在 Android Studio 中重新 Sync → Build → Run
4. 观察识别效果是否与 Codelabs 原版一致